# Bundesliga predictor

In [1]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from rapidfuzz import process, fuzz

In [49]:
def get_table(season: int) -> pd.DataFrame:

    url = f"https://api.openligadb.de/getbltable/bl1/{season}"
    data = requests.get(url).json()

    data

    rows = []
    for team in data:
        rows.append({
            "team_name" : team["teamName"],
            "points" : team["points"],
            "win" : team["won"],
            "draw" : team["draw"],
            "loss" : team["lost"],
            "goals_for" : team["goals"],
            "goals_against" : team["opponentGoals"],
            "goal_diff" : team["goalDiff"]
            })
    table = pd.DataFrame(rows)
    table["position"] = table.index + 1
    return table

The API provided by OpenLigaDB is incorrect for Bundesliga seasons prior to 2023/24, so I had to switch. I found a better API than the previous one, allowing us to implement more stats in our final table!! This api does not provide a league table, but gives detailed of all matches of each matchday. From there, we can compute the wins, losses, draws along with goal difference. 

In [51]:
def get_BL_matches(season: int) -> pd.DataFrame :
    url = f"https://raw.githubusercontent.com/openfootball/football.json/master/{season}-{season%100 + 1}/de.1.json"
    data = requests.get(url).json()

    rows = []
    for match in data["matches"]:
        rows.append({
            "matchday": match["round"],
            "date": match["date"],
            "home": match["team1"],
            "away": match["team2"],
            "score_home": match["score"]["ft"][0],
            "score_away": match["score"]["ft"][1]
        })
    matches = pd.DataFrame(rows)
    return matches

In [52]:
def create_table(season: int) -> pd.DataFrame:

    season_matches = get_BL_matches(season)

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            "home_wins" : statistics["home_wins"],
            "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [53]:
def prep_data(start_date : int, end_date : int): #end date should be 2024
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = create_table(year)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy().set_index("team_name")
        curr_table = season_tables[f"{i +1}"].copy().set_index("team_name")
        relegated = prev_table.tail(3)
        promoted_stats = relegated.mean().to_dict()

        for team, row in curr_table.iterrows():
            if team in prev_table.index:
                features = prev_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]].to_dict()
            else:
                features = {k : promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]}
            feature_rows.append(features)
            target_rows.append(row["position"])
    X_train = pd.DataFrame(feature_rows)
    y_train = pd.Series(target_rows)

    last_feature_rows = []
    last_table = season_tables[f"{end_date}"].copy().set_index("team_name")
    last_relegated = last_table.tail(2)
    last_promoted_stats = last_relegated.mean().to_dict()
    last_teams = last_table.index.tolist()

    last_promoted_teams = ["FC Köln", "Hamburger SV"]
    last_relegated_teams = last_relegated.index.tolist()

    for team in last_teams:
        if team in last_relegated_teams:
            continue
        features = last_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]].to_dict()
        last_feature_rows.append((team, features))
    for team in last_promoted_teams:
        if team not in last_teams:
            feats = {k : last_promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]}
            last_feature_rows.append((team, feats))
    latest_features_df = pd.DataFrame([feats for _, feats in last_feature_rows],
                                      index=[t for t, _ in last_feature_rows])
    return X_train, y_train, latest_features_df

In [156]:
X, y, df = prep_data(2019, 2024)

In [158]:
df

,points,win,draw,loss,goals_for,goals_against,goal_diff,home_wins,home_losses
FC Bayern München,82.0,25.0,7.0,2.0,99.0,32.0,67.0,14.0,1.0
Bayer 04 Leverkusen,69.0,19.0,12.0,3.0,72.0,43.0,29.0,10.0,3.0
Eintracht Frankfurt,60.0,17.0,9.0,8.0,68.0,46.0,22.0,10.0,3.0
Borussia Dortmund,57.0,17.0,6.0,11.0,71.0,51.0,20.0,11.0,3.0
SC Freiburg,55.0,16.0,7.0,11.0,49.0,53.0,-4.0,9.0,5.0
1. FSV Mainz 05,52.0,14.0,10.0,10.0,55.0,43.0,12.0,6.0,3.0
RB Leipzig,51.0,13.0,12.0,9.0,53.0,48.0,5.0,8.0,3.0
SV Werder Bremen,51.0,14.0,9.0,11.0,54.0,57.0,-3.0,5.0,6.0
VfB Stuttgart,50.0,14.0,8.0,12.0,64.0,53.0,11.0,7.0,8.0
Borussia Mönchengladbach,45.0,13.0,6.0,15.0,55.0,57.0,-2.0,7.0,7.0


Now, we start the random forest using Sklearn.

In [116]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])
    model.fit(X, y)
    return model

In [159]:
X, y, df = prep_data(2010, 2024)

In [118]:
model = train_model(X, y)

In [119]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_
exp_positions = prob.dot(classes)
prediction_df = pd.DataFrame({
        "team": df.index,
        "expected_position": exp_positions
        })

In [120]:
prediction_df

,team,expected_position
0,FC Bayern München,1.440000
1,Bayer 04 Leverkusen,5.685274
2,Eintracht Frankfurt,6.424678
3,Borussia Dortmund,6.360707
4,SC Freiburg,7.427531
5,1. FSV Mainz 05,7.943711
6,RB Leipzig,9.105074
7,SV Werder Bremen,9.431840
8,VfB Stuttgart,9.109896
9,Borussia Mönchengladbach,10.359112


In [121]:
prediction_df = prediction_df.sort_values(["expected_position"], ascending=True)
prediction_df = prediction_df.reset_index(drop= True)
prediction_df["position"] = prediction_df.index +1

In [122]:
prediction_df

,team,expected_position,position
0,FC Bayern München,1.440000,1
1,Bayer 04 Leverkusen,5.685274,2
2,Borussia Dortmund,6.360707,3
3,Eintracht Frankfurt,6.424678,4
4,SC Freiburg,7.427531,5
5,1. FSV Mainz 05,7.943711,6
6,RB Leipzig,9.105074,7
7,VfB Stuttgart,9.109896,8
8,SV Werder Bremen,9.431840,9
9,Borussia Mönchengladbach,10.359112,10


I think I could add some more stats. Some useful aspects could be:
- squad price (total cost of squad at start of season)
- squad rating (avg. rating of starting XI) 

We can get this using a kaggle dataset. 

In [13]:
clubs = pd.read_csv("data/clubs.csv")
player_val = pd.read_csv("data/player_valuations.csv")
players = pd.read_csv("data/players.csv")

In [4]:
player_val

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,12359,2004-03-11,250000,RC Lens B,8152,NaN
...,...,...,...,...,...,...
435438,1175607,2026-03-03,100000,Kirklarelispor,27603,NaN
435439,1181950,2026-03-03,10000,Arnavutköy Belediyesi FSK,45493,NaN
435440,1256115,2026-03-03,100000,Karacabey Belediye Spor,20699,NaN
435441,1303292,2026-03-03,25000,Ankara Demirspor,12524,NaN


In [20]:
bvb_players = players[players["current_club_id"] == 16]

bvb_players

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
17,410,Sebastian,Kehl,Sebastian Kehl,2014,16,sebastian-kehl,Germany,Fulda,Germany,...,left,187.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/sebastian-kehl...,L1,Borussia Dortmund,1000000.0,9000000.0
40,1073,Manuel,Friedrich,Manuel Friedrich,2013,16,manuel-friedrich,Germany,Bad Kreuznach,Germany,...,NaN,NaN,NaN,Dr. Michael Becker,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/manuel-friedri...,L1,Borussia Dortmund,500000.0,5500000.0
100,2150,Oliver,Kirch,Oliver Kirch,2014,16,oliver-kirch,Germany,Soest,Germany,...,right,182.0,NaN,Dirk Hebel,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/oliver-kirch/p...,L1,Borussia Dortmund,500000.0,1800000.0
109,2374,Patrick,Owomoyela,Patrick Owomoyela,2012,16,patrick-owomoyela,Germany,Hamburg,Germany,...,right,187.0,NaN,Uwe Kathmann,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/patrick-owomoy...,L1,Borussia Dortmund,200000.0,5000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32562,1009446,Tyler,Meiser,Tyler Meiser,2024,16,tyler-meiser,NaN,NaN,Germany,...,right,189.0,2028-06-30 00:00:00,SBE Management AG,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tyler-meiser/p...,L1,Borussia Dortmund,50000.0,50000.0
32833,1045762,Jordi,Paulina,Jordi Paulina,2024,16,jordi-paulina,Netherlands,Odijk,Curacao,...,right,191.0,2030-06-30 00:00:00,NACR-Sports,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/jordi-paulina/...,L1,Borussia Dortmund,250000.0,250000.0
32989,1058359,Samuele,Inácio,Samuele Inácio,2024,16,samuele-inacio,Italy,Bergamo,Italy,...,right,176.0,2027-06-30 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/samuele-inacio...,L1,Borussia Dortmund,NaN,NaN
33208,1074990,Luca,Reggiani,Luca Reggiani,2025,16,luca-reggiani,Italy,Modena,Italy,...,right,194.0,2027-06-30 00:00:00,TMP SOCCER srl,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/luca-reggiani/...,L1,Borussia Dortmund,1000000.0,1000000.0


In [19]:
dortmund_vals = player_val[player_val["current_club_id"] == 16]

dortmund_vals

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
21,26,2004-10-04,1500000,Borussia Dortmund,16,L1
36,410,2004-10-04,5000000,Borussia Dortmund,16,L1
59,1073,2004-10-04,1200000,1.FSV Mainz 05,16,L1
119,2150,2004-10-04,300000,Borussia Mönchengladbach,16,L1
127,2374,2004-10-04,700000,Arminia Bielefeld,16,L1
...,...,...,...,...,...,...
430002,884820,2025-05-06,1000000,Borussia Dortmund,16,L1
431147,505653,2025-09-06,20000000,UD Las Palmas,16,L1
433721,684116,2025-11-06,200000,Borussia Dortmund,16,L1
433733,866579,2025-11-06,1500000,Borussia Dortmund,16,L1


# Adding additional stats

I want to fetch the club id's for all the clubs in the bundesliga over the seasons I'm considering. To do so, I can check which teams are problematic, and then create a manual mapping.

In [15]:
name_mapping = {
    "SC Freiburg" : "Sport-Club Freiburg",
    "1. FSV Mainz 05" : "1. Fußball- und Sportverein Mainz 05",
    "RB Leipzig" : "RasenBallsport Leipzig",
    "SV Werder Bremen" : "Sportverein Werder Bremen von 1899",
    "VfB Stuttgart" : "Verein für Bewegungsspiele Stuttgart 1893",
    "Borussia Mönchengladbach" : "Borussia Verein für Leibesübungen 1900 Mönchengladbach",
    "VfL Wolfsburg" : "Verein für Leibesübungen Wolfsburg",
    "FC Augsburg" : "Fußball-Club Augsburg 1907",
    "1. FC Union Berlin" : "1. Fußballclub Union Berlin",
    "FC St. Pauli 1910" : "Fußball-Club St. Pauli von 1910",
    "TSG 1899 Hoffenheim" : "Turn- und Sportgemeinschaft 1899 Hoffenheim Fußball-Spielbetriebs",
    "1. FC Heidenheim 1846" : "1. Fußballclub Heidenheim 1846",
    "VfL Bochum 1848" : "VfL Bochum",
    "1. FC Köln" : "1. Fußball-Club Köln",
    "SpVgg Greuther Fürth 1903" : "SpVgg Greuther Fürth",
    "Hamburger SV" : "Hamburger Sport Verein",
    "1. FC Nürnberg" : "1.FC Nuremberg",
    "FC St. Pauli" : "Fußball-Club St. Pauli von 1910",
    "Bayer Leverkusen" : "Bayer 04 Leverkusen Fußball",
    "Bor. Mönchengladbach" : "Borussia Verein für Leibesübungen 1900 Mönchengladbach",
    "1. FC Kaiserslautern" : "test",
    "Bayern München" : "FC Bayern München"
}

teams: Dict[str, int] = {}

for year in range(2010, 2024):
    table = create_table(year)
    for team in table["team_name"]:
        lookup_name = name_mapping.get(team, team)
        
        result = clubs.loc[clubs["name"].str.contains(lookup_name, case=False), "club_id"]
        
        if result.empty:
            print(f"Could not find club_id for: {team}")
            teams[f"{team}"]= np.nan
        else:
            teams[f"{team}"]= result.values[0]

print(teams)

Could not find club_id for: 1. FC Kaiserslautern
Could not find club_id for: 1. FC Kaiserslautern
{'Borussia Dortmund': np.int64(16), 'Bayer Leverkusen': np.int64(15), 'Bayern München': np.int64(27), 'Hannover 96': np.int64(42), '1. FSV Mainz 05': np.int64(39), '1. FC Nürnberg': np.int64(4), '1. FC Kaiserslautern': nan, 'Hamburger SV': np.int64(41), 'SC Freiburg': np.int64(60), '1. FC Köln': np.int64(3), '1899 Hoffenheim': np.int64(533), 'VfB Stuttgart': np.int64(79), 'Werder Bremen': np.int64(86), 'FC Schalke 04': np.int64(33), 'VfL Wolfsburg': np.int64(82), 'Bor. Mönchengladbach': np.int64(18), 'Eintracht Frankfurt': np.int64(24), 'FC St. Pauli': np.int64(35), 'FC Augsburg': np.int64(167), 'Hertha BSC': np.int64(44), 'Fortuna Düsseldorf': np.int64(38), 'SpVgg Greuther Fürth': np.int64(65), 'Eintracht Braunschweig': np.int64(23), 'SC Paderborn 07': np.int64(127), 'FC Ingolstadt 04': np.int64(4795), 'SV Darmstadt 98': np.int64(105), 'RB Leipzig': np.int64(23826), '1. FC Union Berlin': 

In [16]:
clubs

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,9,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,1003,leicester-city,Leicester City,GB1,NaN,29,25.8,17,58.6,10,King Power Stadium,32259,+€57.30m,NaN,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
3,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.0,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...
4,1010,fc-watford,Watford FC,GB1,NaN,30,26.3,24,80.0,12,Vicarage Road,21577,+€42.02m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/fc-watford/sta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,985,manchester-united,Manchester United Football Club,GB1,NaN,26,25.9,19,73.1,14,Old Trafford,74879,€-176.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/manchester-uni...
447,987,motherwell-fc,Motherwell Football Club,SC1,NaN,24,26.9,13,54.2,3,Fir Park,13677,+€5.22m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/motherwell-fc/...
448,989,afc-bournemouth,Association Football Club Bournemouth,GB1,NaN,26,25.4,20,76.9,12,Vitality Stadium,11307,+€130.31m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/afc-bournemout...
449,993,fc-cordoba,Córdoba CF,ES1,NaN,24,27.9,4,16.7,0,Nuevo Arcángel,21822,+-0,NaN,2014,../data/raw/transfermarkt-scraper/2014/clubs.j...,https://www.transfermarkt.co.uk/fc-cordoba/sta...


In [17]:
L1 = clubs[clubs["domestic_competition_id"] == "L1"]

L1

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
14,105,sv-darmstadt-98,SV Darmstadt 98,L1,NaN,27,25.6,13,48.1,1,Merck-Stadion am Böllenfalltor,17810,+€3.05m,NaN,2023,../data/raw/transfermarkt-scraper/2023/clubs.j...,https://www.transfermarkt.co.uk/sv-darmstadt-9...
65,127,sc-paderborn-07,SC Paderborn 07,L1,NaN,26,25.7,5,19.2,1,Home Deluxe Arena,15000,€-450k,NaN,2019,../data/raw/transfermarkt-scraper/2019/clubs.j...,https://www.transfermarkt.co.uk/sc-paderborn-0...
95,15,bayer-04-leverkusen,Bayer 04 Leverkusen Fußball,L1,NaN,26,26.7,21,80.8,14,BayArena,30210,+€32.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/bayer-04-lever...
103,16,borussia-dortmund,Borussia Dortmund,L1,NaN,25,26.5,13,52.0,12,SIGNAL IDUNA PARK,81365,€-23.20m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/borussia-dortm...
109,167,fc-augsburg,Fußball-Club Augsburg 1907,L1,NaN,28,25.5,19,67.9,9,WWK ARENA,30660,€-8.25m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/fc-augsburg/st...
115,18,borussia-monchengladbach,Borussia Verein für Leibesübungen 1900 Mönchen...,L1,NaN,28,25.8,15,53.6,13,Stadion im Borussia-Park,54042,+€10.65m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/borussia-monch...
130,2036,1-fc-heidenheim-1846,1. Fußballclub Heidenheim 1846,L1,NaN,29,26.2,7,24.1,3,Voith-Arena,15000,+€7.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/1-fc-heidenhei...
143,23,eintracht-braunschweig,Eintracht Braunschweig,L1,NaN,30,25.6,11,36.7,2,EINTRACHT-Stadion,23325,+€750k,NaN,2013,../data/raw/transfermarkt-scraper/2013/clubs.j...,https://www.transfermarkt.co.uk/eintracht-brau...
152,23826,rasenballsport-leipzig,RasenBallsport Leipzig,L1,NaN,31,23.4,21,67.7,13,Red Bull Arena,47069,+€38.05m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/rasenballsport...


In [14]:
appearances = pd.read_csv("data/appearances.csv")

appearances

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,6646,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1734618,4649338_706618,4649338,706618,232,232,2026-03-02,Vladislav Saus,RU1,0,0,0,0,7
1734619,4649338_756991,4649338,756991,232,232,2026-03-02,Pablo Solari,RU1,1,0,1,0,90
1734620,4649338_793774,4649338,793774,41231,41231,2026-03-02,Marcelo Alves,RU1,0,0,0,0,90
1734621,4649338_811544,4649338,811544,41231,41231,2026-03-02,Aleksandr Kovalenko,RU1,0,0,0,0,1


In [16]:
valuations = pd.read_csv("data/player_valuations.csv")
print(valuations.columns.tolist())
valuations.head()

['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,12359,2004-03-11,250000,RC Lens B,8152,NaN


In [17]:
# get values around the start of the 2015/16 season
start_of_season = valuations[
    (valuations["date"] >= "2015-07-01") & 
    (valuations["date"] <= "2015-09-01")
]

statss = start_of_season.groupby("current_club_id")["market_value_in_eur"].sum()

In [ ]:
statss.head(3)

This data isn't complete enough for the smaller clubs. I fetched data from Transfermarkt for all Bundesliga teams.

### Using Transfermarkt data for squad values at start of season

We can now add a squad value column to our data. We begin by slightly cleaning the data in order for the club values are numeric only. Then, we can take the same steps as before. 

In [57]:
table_2024 = create_table(2024)

In [5]:
def transform_value(col):
    if "m" in col:
        col = float(col.replace("m", "")) * 1000000
        return col
    elif "k" in col:
        col = float(col.replace("k", "")) * 1000
        return col
    return col

def adjust_values(stats : pd.DataFrame) -> pd.DataFrame :
    # clean data
    stats["value"] = stats["value"].astype(str).str.replace("€", "")
    stats["value"] = stats["value"].astype(str).str.replace(",", ".")
    stats["value"] = stats["value"].apply(transform_value)
    #make value relative
    total_budget = stats["value"].sum()
    stats["rel_value"] = stats["value"] / total_budget
    #get avg player value
    stats["avg_player_value"] = stats["value"] / stats["size"]
    #drop uninteresting columns
    stats.drop(columns= "foreigners", inplace= True)
    stats.drop(columns= "size", inplace= True)
    stats.drop(columns= "value", inplace= True)
    return stats

def add_pre_season_stats(season : int, table : pd.DataFrame, key_left : str) -> pd.DataFrame :
    stats = pd.read_csv(f"data/squad_values_{season}-{season +1}.csv")
    #fuzzy matching - the names from the table are not necessarily the same as the ones in the stats
    teams_left = table[key_left].tolist()

    def match_team(teams_right):
        match, score, _ = process.extractOne(
            teams_right,
            teams_left,
            scorer = fuzz.token_sort_ratio
        )
        return match if score > 60 else None
    
    # print(f"before: {stats}")

    stats["team"] = stats["team"].apply(match_team)

    # print(f"after: {stats}")

    result = pd.merge(table, stats, left_on= key_left, right_on="team", how= "left")

    result.drop(columns="team", inplace= True)

    return result

## Prepping the data using these new stats

In [64]:
stats = pd.read_csv("data/squad_values_2025-2026.csv")
stats

,team,size,age,foreigners,value
0,Bayern Munich,24,27.1,12,€959.95m
1,Borussia Dortmund,25,26.5,13,€484.40m
2,RB Leipzig,31,23.4,21,€434.40m
3,Bayer 04 Leverkusen,27,26.4,21,€422.75m
4,Eintracht Frankfurt,29,25.4,19,€390.05m
5,VfB Stuttgart,30,24.7,15,€341.40m
6,VfL Wolfsburg,33,25.3,24,€249.60m
7,TSG 1899 Hoffenheim,29,25.3,20,€197.45m
8,SV Werder Bremen,30,25.2,18,€191.00m
9,SC Freiburg,25,27.9,9,€171.20m


In [65]:
stats["value"].sum()

'€959.95m€484.40m€434.40m€422.75m€390.05m€341.40m€249.60m€197.45m€191.00m€171.20m€149.80m€146.38m€145.98m€142.80m€136.45m€122.65m€66.35m€63.55m'

In [145]:
# We can use the fact that we will be using features from previous seasons along with features from before the season start

LAST_SEASON_FEATURES = ["points", "win", "draw", "loss", "goals_for", 
                         "goals_against", "goal_diff", "home_wins", "home_losses"]
PRESEASON_FEATURES = ["value", "size", "age"]

def prep_data2(start_date : int, end_date : int): #end date should be 2024 (end_date is the starting year of the last fully played season)
    # we prepare the training data for seasons prior to the current season
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = create_table(year)
        # table = add_pre_season_stats(year +1, table)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy()
        curr_table = season_tables[f"{i +1}"].copy()

        prev_table[LAST_SEASON_FEATURES] = prev_table[LAST_SEASON_FEATURES].astype(float)
        curr_table[LAST_SEASON_FEATURES] = curr_table[LAST_SEASON_FEATURES].astype(float)
        # prev_table = season_tables[f"{i}"].copy().set_index("team_name")  #stats from previous season
        # curr_table = season_tables[f"{i +1}"].copy().set_index("team_name") #stats from next season
        prev_stats = pd.read_csv(f"data/squad_values_{i}-{i+1}.csv").set_index("team")
        curr_stats = pd.read_csv(f"data/squad_values_{i+1}-{i+2}.csv").set_index("team")
        
        prev_teams = set(prev_stats.index)
        curr_teams = set(curr_stats.index)
        promoted_teams = curr_teams - prev_teams
        promoted_teams = list(promoted_teams)
        n_promoted = len(promoted_teams)

        prev_table.iloc[-n_promoted:, prev_table.columns.isin(LAST_SEASON_FEATURES)] = prev_table.tail(n_promoted)[LAST_SEASON_FEATURES].mean().values 
        
        #we change the names of the teams in the last two rows to fit the names of the newly promoted teams
        for index, team in enumerate(promoted_teams):
            prev_table.iloc[[-n_promoted + index], prev_table.columns.get_loc("team_name")] = promoted_teams[index]
        prev_table = add_pre_season_stats(i+1, prev_table, "team_name")
        prev_table = adjust_values(prev_table)
        prev_table = prev_table.set_index("team_name")

        # Append features (all columns except position)
        feature_rows.append(prev_table.drop(columns=["position"]))

        curr_table = curr_table.set_index("team_name")
        target_rows.append(curr_table["position"])  # position in the NEXT season is the target

    X_train = pd.concat(feature_rows, axis=0).reset_index(drop=True)
    y_train = pd.concat(target_rows, axis=0).reset_index(drop=True)

    #     relegated = prev_table.tail(n_promoted)
    #     promoted_stats = relegated.mean().to_dict()

    #     for team, row in curr_table.iterrows():
    #         if team in prev_table.index: # for teams that were not relegated
    #             features = prev_table.loc[team][LAST_SEASON_FEATURES + PRESEASON_FEATURES].to_dict() #we use the table from the previous year
    #         else:
    #             features = {k : promoted_stats[k] for k in LAST_SEASON_FEATURES + PRESEASON_FEATURES} 
    #         feature_rows.append(features)
    #         target_rows.append(row["position"]) #important to note that row is in curr_table --> we are assigning the position the following season
    # X_train = pd.DataFrame(feature_rows)
    # y_train = pd.Series(target_rows)

    last_table = create_table(end_date)
    last_table[LAST_SEASON_FEATURES] = last_table[LAST_SEASON_FEATURES].astype(float) # we will be taking avg, so no longer integer valued
    
    #since we dont actually have stats for prmoted teams, we take avg stats of the relegated teams
    last_table.iloc[-2:, last_table.columns.isin(LAST_SEASON_FEATURES)] = last_table.tail(2)[LAST_SEASON_FEATURES].mean().values 
    
    #we change the names of the teams in the last two rows to fit the names of the newly promoted teams
    last_table.loc[last_table["team_name"] == "Holstein Kiel", "team_name"] = "FC Köln"
    last_table.loc[last_table["team_name"] == "VfL Bochum 1848", "team_name"] = "Hamburger SV"

    last_table = add_pre_season_stats(end_date + 1, last_table, "team_name") #we are adding the pre_season stats for the following season, hence the end_date +1
    last_table = adjust_values(last_table)

    latest_features_df = last_table.set_index("team_name")
    latest_features_df = latest_features_df.drop(columns= ["position"])
    return X_train, y_train, latest_features_df

In [146]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
        ))
    ])
    model.fit(X, y)
    return model

In [147]:
X, y, df = prep_data2(2019, 2024)

In [148]:
model = train_model(X, y)

prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_
exp_positions = prob.dot(classes)
prediction_df = pd.DataFrame({
        "team": df.index,
        "expected_position": exp_positions
        })

prediction_df.sort_values(["expected_position"], ascending=True, inplace= True)
prediction_df
prediction_df = prediction_df.reset_index(drop= True)
prediction_df
prediction_df["position"] = prediction_df.index +1

In [149]:
prediction_df

,team,expected_position,position
0,FC Bayern München,1.150000,1
1,Bayer 04 Leverkusen,3.201857,2
2,Borussia Dortmund,4.449000,3
3,Eintracht Frankfurt,4.521389,4
4,RB Leipzig,6.237917,5
5,1. FSV Mainz 05,6.686407,6
6,VfB Stuttgart,6.702214,7
7,SC Freiburg,6.781389,8
8,SV Werder Bremen,7.980335,9
9,Borussia Mönchengladbach,9.605496,10


In [89]:
importances = model.named_steps["rf"].feature_importances_

pd.Series(importances, index=X.columns).sort_values(ascending=False)

points              0.159873
goal_diff           0.122369
win                 0.122293
avg_player_value    0.090834
goals_against       0.087094
rel_value           0.075025
goals_for           0.073254
loss                0.070464
age                 0.061429
home_wins           0.057736
home_losses         0.044325
draw                0.035303
dtype: float64

In [ ]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_

# positions
zones = {
    "Champion":      classes <= 1,
    "2_4":      (classes >= 2) & (classes <= 4),
    "5_10":   (classes >= 5) & (classes <= 10),
    "11_17": (classes >= 11) & (classes <= 17),
    "Relegated": classes >= 18
}

prediction_df = pd.DataFrame({"team": df.index})
prediction_df["expected_position"] = prob.dot(classes)

for zone, mask in zones.items():
    prediction_df[f"prob_{zone}"] = prob[:, mask].sum(axis=1)

prediction_df = prediction_df.sort_values("expected_position")

In [178]:
prediction_df

,team,expected_position,prob_title,prob_2_4,prob_5_10,prob_11_17,prob_relegation
0,FC Bayern München,1.170000,0.91,0.090000,0.000000,0.000000,0.000000
1,Bayer 04 Leverkusen,3.375000,0.07,0.630000,0.300000,0.000000,0.000000
3,Borussia Dortmund,4.545238,0.00,0.506667,0.493333,0.000000,0.000000
2,Eintracht Frankfurt,4.624071,0.00,0.272000,0.728000,0.000000,0.000000
5,1. FSV Mainz 05,6.507619,0.00,0.040000,0.951429,0.008571,0.000000
4,SC Freiburg,6.538488,0.00,0.080000,0.853500,0.066500,0.000000
6,RB Leipzig,6.754961,0.00,0.010000,0.969567,0.020433,0.000000
8,VfB Stuttgart,7.056914,0.00,0.010000,0.930341,0.059659,0.000000
7,SV Werder Bremen,7.878345,0.00,0.000000,0.915179,0.084821,0.000000
10,VfL Wolfsburg,9.780656,0.00,0.000000,0.658559,0.341441,0.000000


In [ ]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_

# positions
zones = {
    "Champion":      classes <= 1,
    "2_4":      (classes >= 2) & (classes <= 4),
    "5_10":   (classes >= 5) & (classes <= 10),
    "11_17": (classes >= 11) & (classes <= 17),
    "Relegated": classes >= 18
}

prediction_df = pd.DataFrame({"team": df.index})
prediction_df["expected_position"] = prob.dot(classes)

for zone, mask in zones.items():
    prediction_df[f"prob_{zone}"] = prob[:, mask].sum(axis=1)

prediction_df = prediction_df.sort_values("expected_position")

In [102]:
prediction_df

,team,expected_position,prob_Champion,prob_2_4,prob_5_10,prob_11_17,prob_Relegated
0,FC Bayern München,1.120000,0.95,0.050,0.000000,0.000000,0.000000
1,Bayer 04 Leverkusen,3.285000,0.08,0.680,0.240000,0.000000,0.000000
3,Borussia Dortmund,4.345000,0.01,0.585,0.403571,0.001429,0.000000
2,Eintracht Frankfurt,4.585079,0.00,0.315,0.685000,0.000000,0.000000
6,RB Leipzig,6.488940,0.00,0.040,0.936250,0.023750,0.000000
5,1. FSV Mainz 05,6.682923,0.00,0.025,0.973750,0.001250,0.000000
8,VfB Stuttgart,6.718206,0.00,0.040,0.927639,0.032361,0.000000
4,SC Freiburg,6.732018,0.00,0.020,0.944405,0.035595,0.000000
7,SV Werder Bremen,7.995497,0.00,0.010,0.898396,0.091604,0.000000
9,Borussia Mönchengladbach,9.672626,0.00,0.000,0.693198,0.306802,0.000000


In [11]:
stats = stats.dropna()

In [12]:
stats

,club,years_active,size,age,foreigners,value
0,Bayern Munich,61,24.0,27.1,12.0,€959.95m
1,Borussia Dortmund,59,25.0,26.5,13.0,€484.40m
2,Werder Bremen,61,30.0,25.2,18.0,€191.00m
3,VfB Stuttgart,59,30.0,24.7,15.0,€341.40m
4,Borussia Mönchengladbach,58,28.0,25.8,15.0,€149.80m
5,Hamburger SV,55,31.0,24.7,23.0,€145.98m
6,Eintracht Frankfurt f,57,29.0,25.4,19.0,€390.05m
8,Bayer Leverkusen,47,27.0,26.4,21.0,€422.75m
9,1. FC Köln,52,30.0,25.7,12.0,€136.45m
13,VfL Wolfsburg,29,33.0,25.3,24.0,€249.60m


# Accurate longevity stats for each season

The idea here is to get longevity for each season, starting from the 2025/26 season and downwards. Maybe not the most efficient method, but we can take the table from previous years, and subtract 1 for each team present. 

In [104]:
def create_longevity_data(start_year):
    longevity = pd.read_csv("data/bundesliga_longevity_2025-2026.csv")
    clubs = longevity["club"].tolist()
    for year in range(2024, start_year -1, -1):
        table = create_table(year)
        teams = table["team_name"].tolist()
        matched_teams = []
        for team in teams:
            match, score, _ = process.extractOne(
                team,
                clubs,
                scorer=fuzz.token_sort_ratio
            )
            if score > 60:
                matched_teams.append(match)

        longevity.loc[longevity["club"].isin(matched_teams),"years_active"] -= 1
        longevity.to_csv(f"data/bundesliga_longevity_{year}-{year + 1}.csv", index= False)
    return 0

In [105]:
create_longevity_data(2019)

0

# Adding longevity as a factor

Let's see if longevity can play a part in ranking. A team that's been many seasons in the Bundesliga may be more inclined to have stronger results than a relatively new team with little Bundesliga experience. Let's see if this plays a part and can increase accuracy. 

In [106]:
def merge_on_team_name(table_left : pd.DataFrame, key_left : str, table_right : pd.DataFrame, key_right : str) -> pd.DataFrame :
    # stats = pd.read_csv(f"data/squad_values_{season}-{season +1}.csv")
    #fuzzy matching - the names from the table are not necessarily the same as the ones in the stats
    teams_left = table_left[key_left].tolist()

    def match_team(teams_right):
        match, score, _ = process.extractOne(
            teams_right,
            teams_left,
            scorer = fuzz.token_sort_ratio
        )
        # if score <= 60:
        #     # print(f"LOW SCORE: '{teams_right}' -> '{match}' ({score})")
        #     return teams_right

        # if teams_right != match:
        #     print(f"MATCH: '{teams_right}' -> '{match}' ({score})")

        # return match
        return match if score > 65 else teams_right
    
    # print(f"before: {stats}")

    table_right[key_right] = table_right[key_right].apply(match_team)

    # print(f"after: {stats}")

    result = pd.merge(table_left, table_right, left_on= key_left, right_on= key_right, how= "left")

    result.drop(columns= key_right, inplace= True)

    return result

# key_right is team
#key_left is team_name

In [111]:
# We can use the fact that we will be using features from previous seasons along with features from before the season start

LAST_SEASON_FEATURES = ["points", "win", "draw", "loss", "goals_for", 
                         "goals_against", "goal_diff", "home_wins", "home_losses"]
PRESEASON_FEATURES = ["value", "size", "age", "years_active"]

def prep_data3(start_date : int, end_date : int): #end date should be 2024 (end_date is the starting year of the last fully played season)
    # we prepare the training data for seasons prior to the current season
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = create_table(year)
        # table = add_pre_season_stats(year +1, table)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy()
        curr_table = season_tables[f"{i +1}"].copy()

        prev_table[LAST_SEASON_FEATURES] = prev_table[LAST_SEASON_FEATURES].astype(float)
        curr_table[LAST_SEASON_FEATURES] = curr_table[LAST_SEASON_FEATURES].astype(float)
        # prev_table = season_tables[f"{i}"].copy().set_index("team_name")  #stats from previous season
        # curr_table = season_tables[f"{i +1}"].copy().set_index("team_name") #stats from next season
        prev_stats = pd.read_csv(f"data/squad_values_{i}-{i+1}.csv").set_index("team")
        curr_stats = pd.read_csv(f"data/squad_values_{i+1}-{i+2}.csv").set_index("team")
        
        prev_teams = set(prev_stats.index)
        curr_teams = set(curr_stats.index)
        promoted_teams = curr_teams - prev_teams
        promoted_teams = list(promoted_teams)
        n_promoted = len(promoted_teams)

        prev_table.iloc[-n_promoted:, prev_table.columns.isin(LAST_SEASON_FEATURES)] = prev_table.tail(n_promoted)[LAST_SEASON_FEATURES].mean().values 
        
        #we change the names of the teams in the last two rows to fit the names of the newly promoted teams
        for index, team in enumerate(promoted_teams):
            prev_table.iloc[[-n_promoted + index], prev_table.columns.get_loc("team_name")] = promoted_teams[index]
        
        curr_stats = pd.read_csv(f"data/squad_values_{i+1}-{i+2}.csv")
        curr_longevity = pd.read_csv(f"data/bundesliga_longevity_{i+1}-{i+2}.csv")

        curr_stats = merge_on_team_name(curr_stats, "team", curr_longevity, "club") #we merge the longevity and pre-season stats together 
        prev_table = merge_on_team_name(prev_table, "team_name", curr_stats, "team") #we add the stats from the current season

        
        prev_table = adjust_values(prev_table)

        prev_table = prev_table.set_index("team_name")

        # Append features (all columns except position)
        feature_rows.append(prev_table.drop(columns=["position"]))

        curr_table = curr_table.set_index("team_name")
        target_rows.append(curr_table["position"])  # position in the NEXT season is the target

    X_train = pd.concat(feature_rows, axis=0).reset_index(drop=True)
    y_train = pd.concat(target_rows, axis=0).reset_index(drop=True)

    #     relegated = prev_table.tail(n_promoted)
    #     promoted_stats = relegated.mean().to_dict()

    #     for team, row in curr_table.iterrows():
    #         if team in prev_table.index: # for teams that were not relegated
    #             features = prev_table.loc[team][LAST_SEASON_FEATURES + PRESEASON_FEATURES].to_dict() #we use the table from the previous year
    #         else:
    #             features = {k : promoted_stats[k] for k in LAST_SEASON_FEATURES + PRESEASON_FEATURES} 
    #         feature_rows.append(features)
    #         target_rows.append(row["position"]) #important to note that row is in curr_table --> we are assigning the position the following season
    # X_train = pd.DataFrame(feature_rows)
    # y_train = pd.Series(target_rows)

    last_table = create_table(end_date)
    last_table[LAST_SEASON_FEATURES] = last_table[LAST_SEASON_FEATURES].astype(float) # we will be taking avg, so no longer integer valued
    
    #since we dont actually have stats for prmoted teams, we take avg stats of the relegated teams
    last_table.iloc[-2:, last_table.columns.isin(LAST_SEASON_FEATURES)] = last_table.tail(2)[LAST_SEASON_FEATURES].mean().values 
    
    #we change the names of the teams in the last two rows to fit the names of the newly promoted teams
    last_table.loc[last_table["team_name"] == "Holstein Kiel", "team_name"] = "FC Köln"
    last_table.loc[last_table["team_name"] == "VfL Bochum 1848", "team_name"] = "Hamburger SV"

    last_stats = pd.read_csv(f"data/squad_values_{end_date+1}-{end_date+2}.csv")
    last_longevity = pd.read_csv(f"data/bundesliga_longevity_{end_date+1}-{end_date+2}.csv")

    last_stats = merge_on_team_name(last_stats, "team", last_longevity, "club") #we merge the longevity and pre-season stats together 
    last_table = merge_on_team_name(last_table, "team_name", last_stats, "team") #we add the stats from the current season

    last_table = adjust_values(last_table)

    latest_features_df = last_table.set_index("team_name")
    latest_features_df = latest_features_df.drop(columns= ["position"])
    return X_train, y_train, latest_features_df

In [112]:
X, y, df = prep_data3(2019, 2024)
print(X.shape, y.shape)

(90, 13) (90,)


In [140]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
        ))
    ])
    model.fit(X, y)
    return model

functions = [prep_data, prep_data2, prep_data3]

tables = {}
count = 1
for f in functions:        
        X, y, df = f(2019, 2024)
        model = train_model(X, y)

        prob = model.predict_proba(df)
        classes = model.named_steps["rf"].classes_
        exp_positions = prob.dot(classes)
        prediction_df = pd.DataFrame({
                "team": df.index,
                "expected_position": exp_positions
                })

        prediction_df.sort_values(["expected_position"], ascending=True, inplace= True)
        prediction_df
        prediction_df = prediction_df.reset_index(drop= True)
        prediction_df
        prediction_df["position"] = prediction_df.index +1

        tables[f"model_{count}"] = [prediction_df]
        
        prob = model.predict_proba(df)
        classes = model.named_steps["rf"].classes_

        # positions
        zones = {
            "Champion":      classes <= 1,
            "2_4":      (classes >= 2) & (classes <= 4),
            "5_10":   (classes >= 5) & (classes <= 10),
            "11_17": (classes >= 11) & (classes <= 17),
            "Relegated": classes >= 18
        }

        prediction_df = pd.DataFrame({"team": df.index})
        prediction_df["expected_position"] = prob.dot(classes)

        for zone, mask in zones.items():
            prediction_df[f"prob_{zone}"] = prob[:, mask].sum(axis=1)

        prediction_df = prediction_df.sort_values("expected_position")

        tables[f"model_{count}"].append(prediction_df)
        count+=1

# Model comparisons

Let's compare how are models performed.

## Model 1

This model only uses the table from year $N$ to predict year $N+1$.

In [133]:
tables["model_1"][0]

,team,expected_position,position
0,FC Bayern München,1.749429,1
1,Bayer 04 Leverkusen,4.472942,2
2,1. FSV Mainz 05,6.450508,3
3,RB Leipzig,6.762026,4
4,Eintracht Frankfurt,6.864053,5
5,Borussia Dortmund,7.216996,6
6,VfB Stuttgart,8.180825,7
7,SC Freiburg,8.934958,8
8,SV Werder Bremen,9.101592,9
9,Borussia Mönchengladbach,10.211546,10


In [134]:
tables["model_1"][1]

,team,expected_position,prob_Champion,prob_2_4,prob_5_10,prob_11_17,prob_Relegated
0,FC Bayern München,1.749429,0.586857,0.357714,0.055429,0.000000,0.000000
1,Bayer 04 Leverkusen,4.472942,0.031973,0.676848,0.194678,0.096500,0.000000
5,1. FSV Mainz 05,6.450508,0.038360,0.316517,0.399196,0.244258,0.001669
6,RB Leipzig,6.762026,0.036155,0.234955,0.513845,0.213868,0.001176
2,Eintracht Frankfurt,6.864053,0.021455,0.386848,0.353058,0.238638,0.000000
3,Borussia Dortmund,7.216996,0.038917,0.297660,0.313492,0.349931,0.000000
8,VfB Stuttgart,8.180825,0.270894,0.026424,0.298724,0.403244,0.000714
4,SC Freiburg,8.934958,0.069759,0.131395,0.331769,0.462863,0.004215
7,SV Werder Bremen,9.101592,0.121199,0.146527,0.289551,0.440429,0.002294
9,Borussia Mönchengladbach,10.211546,0.026653,0.019018,0.478155,0.471928,0.004245


## Model 2

This model incorporates squad values for current season, roster size, and average age.

In [141]:
tables["model_2"][0]

,team,expected_position,position
0,FC Bayern München,1.150000,1
1,Bayer 04 Leverkusen,3.201857,2
2,Borussia Dortmund,4.449000,3
3,Eintracht Frankfurt,4.521389,4
4,RB Leipzig,6.237917,5
5,1. FSV Mainz 05,6.686407,6
6,VfB Stuttgart,6.702214,7
7,SC Freiburg,6.781389,8
8,SV Werder Bremen,7.980335,9
9,Borussia Mönchengladbach,9.605496,10


In [142]:
tables["model_2"][1]

,team,expected_position,prob_Champion,prob_2_4,prob_5_10,prob_11_17,prob_Relegated
0,FC Bayern München,1.150000,0.920,0.080000,0.000000,0.000000,0.000000
1,Bayer 04 Leverkusen,3.201857,0.092,0.679000,0.229000,0.000000,0.000000
3,Borussia Dortmund,4.449000,0.010,0.605000,0.381071,0.003929,0.000000
2,Eintracht Frankfurt,4.521389,0.000,0.374857,0.625143,0.000000,0.000000
6,RB Leipzig,6.237917,0.000,0.063857,0.933643,0.002500,0.000000
5,1. FSV Mainz 05,6.686407,0.000,0.037857,0.957560,0.004583,0.000000
8,VfB Stuttgart,6.702214,0.000,0.047500,0.894583,0.057917,0.000000
4,SC Freiburg,6.781389,0.000,0.030000,0.906905,0.063095,0.000000
7,SV Werder Bremen,7.980335,0.000,0.012500,0.895000,0.092500,0.000000
9,Borussia Mönchengladbach,9.605496,0.000,0.002500,0.767302,0.230198,0.000000


## Model 3

This model incorporates the team's longevity within the league. 

In [143]:
tables["model_3"][0]

,team,expected_position,position
0,FC Bayern München,1.180000,1
1,Bayer 04 Leverkusen,3.262500,2
2,Eintracht Frankfurt,4.325833,3
3,Borussia Dortmund,4.502667,4
4,RB Leipzig,6.395143,5
5,SC Freiburg,6.829333,6
6,1. FSV Mainz 05,6.868262,7
7,VfB Stuttgart,7.095889,8
8,SV Werder Bremen,8.053310,9
9,Borussia Mönchengladbach,9.828127,10


In [144]:
tables["model_3"][1]

,team,expected_position,prob_Champion,prob_2_4,prob_5_10,prob_11_17,prob_Relegated
0,FC Bayern München,1.180000,0.92,0.080000,0.000000,0.000000,0.000000
1,Bayer 04 Leverkusen,3.262500,0.12,0.615000,0.265000,0.000000,0.000000
2,Eintracht Frankfurt,4.325833,0.00,0.430833,0.569167,0.000000,0.000000
3,Borussia Dortmund,4.502667,0.00,0.475000,0.522500,0.002500,0.000000
6,RB Leipzig,6.395143,0.00,0.040000,0.937571,0.022429,0.000000
4,SC Freiburg,6.829333,0.00,0.085000,0.874286,0.040714,0.000000
5,1. FSV Mainz 05,6.868262,0.00,0.025000,0.962857,0.012143,0.000000
8,VfB Stuttgart,7.095889,0.00,0.030000,0.884960,0.085040,0.000000
7,SV Werder Bremen,8.053310,0.00,0.010000,0.882929,0.107071,0.000000
9,Borussia Mönchengladbach,9.828127,0.00,0.010000,0.722133,0.267867,0.000000


Strangely, adding the longevity factor seems to weaken the model (from model 2 to model 3). It gives Frankfurt the edge over Dortmund. But looking at the current Bundesliga table, Dortmund is well ahead of Frankfurt. Maybe this last parameter is not so useful in predicting rankings.